While reading dataframes, we have 2 methods

1. Internal - HDFS/ Amazon S3/ ADLS Gen 2/ GCS
2. External - Oracle, Mysql, Snowflake, MongoDB

For this some extra setup is required.

Let's say we have a orders table in mysql, how doe we process in spark?

    - Ingest the datasets from mysql to datalake using Ingestion framework

    - Directly connect to external source using Spark API and process it. But this is not recommended.

The storage should always be datalake

There were 3 modes while reading

1. permissive
2. dropMalformed
3. failFast

Let's dsicuss about Dataframe Writer API

There are 4 modes to write

1. overwrite
2. append
3. ignore
4. errorIfExists

While writing back to a disk, we need to mention the folder location. 

overwrite - It will overwrite if it exists or it will create a new folder

ignore - it will ignore creating the folder if it already exists and it won't write the data as well

append - it will add new data to the already existing folder with new data 

errorIfExists - if folder exists it will throw an error 

In [0]:
orders_schema = "order_id long, order_date string, cust_id long, order_status string"

In [0]:
orders_df = spark.read.format("csv").schema(orders_schema).load('abfss://unitycatalog@databricksstudysa.dfs.core.windows.net/catalog/orders_1gb.csv')

In [0]:
orders_df.show()

In [0]:
from pyspark.sql.functions import spark_partition_id
orders_df.select(spark_partition_id()).distinct().count()

In [0]:
# Writing to a UC managed table avoids direct ABFS key requirements
# Change mode to "append" / "ignore" / "errorIfExists" to test the other modes
orders_df.write.format("delta").mode("overwrite").saveAsTable("azure_data_engineering_learn_ws.default.orders_output")

In [0]:
for f in spark.table("azure_data_engineering_learn_ws.default.orders_output").inputFiles():
    display(f)

In [0]:
orders_df.write.mode("overwrite").saveAsTable("azure_data_engineering_learn_ws.default.orders_output_parquet")

In [0]:
for f in spark.table("azure_data_engineering_learn_ws.default.orders_output_parquet").inputFiles():
    display(f)

In [0]:
orders_df.write.mode("error").saveAsTable("azure_data_engineering_learn_ws.default.orders_output_parquet")

In [0]:
orders_df.write.mode("ignore").saveAsTable("azure_data_engineering_learn_ws.default.orders_output_parquet")

In [0]:
orders_df.write.mode("append").saveAsTable("azure_data_engineering_learn_ws.default.orders_output_parquet")

In [0]:
for f in spark.table("azure_data_engineering_learn_ws.default.orders_output_parquet").inputFiles():
    display(f)

In [0]:
df = spark.table("azure_data_engineering_learn_ws.default.orders_output_parquet").inputFiles()


In [0]:
display(df)

In [0]:
len(df)